# Customer Segmentation Project — Car Sales

**Goal:** Segment customers based on purchasing behavior using K-Means clustering, analyze purchase patterns, and visualize the resulting segments.

**Note on the data:** `Car_Sales_Dataset.csv` is a B2B sales log (Brand → Dealer), with no individual end-consumer ID. So here, each **(Dealer, Region)** combination is treated as a *customer* (a dealership buying channel) and segmented on its purchasing behavior — total spend, order frequency, price tier, discount usage, and fuel-type diversity. This is a standard approach for B2B / channel analytics.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('Car_Sales_Dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 2. Build Customer (Dealer × Region) Feature Table

For each dealer-region pair we compute RFM-style behavioral features:
- **Total_Revenue** — monetary value
- **Num_Transactions** — purchase frequency
- **Avg_Unit_Price / Avg_Order_Value** — price tier
- **Avg_Discount** — discount sensitivity
- **FuelType_Diversity / Brand_Diversity** — breadth of purchasing

In [ ]:
grp = df.groupby(['Dealer', 'Region'])

features = grp.agg(
    Total_Revenue=('Revenue (INR)', 'sum'),
    Total_Units=('Units Sold', 'sum'),
    Num_Transactions=('Sale ID', 'count'),
    Avg_Unit_Price=('Unit Price (INR)', 'mean'),
    Avg_Discount=('Discount (%)', 'mean'),
    FuelType_Diversity=('Fuel Type', pd.Series.nunique),
    Brand_Diversity=('Brand', pd.Series.nunique),
).reset_index()

features['Avg_Order_Value'] = features['Total_Revenue'] / features['Num_Transactions']
features['Customer'] = features['Dealer'] + ' - ' + features['Region']
features.head(10)

## 3. Scale Features

In [ ]:
cluster_cols = ['Total_Revenue', 'Total_Units', 'Num_Transactions',
                'Avg_Unit_Price', 'Avg_Discount', 'FuelType_Diversity', 'Avg_Order_Value']

X = features[cluster_cols].values
X_scaled = StandardScaler().fit_transform(X)

## 4. Find Optimal K (Elbow + Silhouette Score)

In [ ]:
inertias, sil_scores = [], []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(K_range), inertias, marker='o'); axes[0].set_title('Elbow Method')
axes[1].plot(list(K_range), sil_scores, marker='o', color='orange'); axes[1].set_title('Silhouette Score')
plt.tight_layout(); plt.show()

print('Best k by silhouette:', list(K_range)[int(np.argmax(sil_scores))])

## 5. Fit K-Means (k=3) & Label Segments

In [ ]:
FINAL_K = 3
kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
features['Segment'] = kmeans.fit_predict(X_scaled)

seg_order = features.groupby('Segment')['Total_Revenue'].mean().sort_values(ascending=False).index
rank_map = {seg: name for seg, name in zip(seg_order, ['High-Value', 'Mid-Value', 'Low-Value'])}
features['Segment_Label'] = features['Segment'].map(rank_map)
features['Segment_Label'].value_counts()

## 6. Visualize Clusters with PCA

In [ ]:
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)
features['PC1'], features['PC2'] = pcs[:, 0], pcs[:, 1]

plt.figure(figsize=(7, 5.5))
palette = {'High-Value': '#2E7D32', 'Mid-Value': '#F9A825', 'Low-Value': '#C62828'}
sns.scatterplot(data=features, x='PC1', y='PC2', hue='Segment_Label', palette=palette,
                s=140, edgecolor='black')
plt.title('Customer Segments (PCA Projection)')
plt.show()

## 7. Segment Profiles (Average Behavior per Segment)

In [ ]:
profile = features.groupby('Segment_Label')[cluster_cols].mean().reindex(
    ['High-Value', 'Mid-Value', 'Low-Value'])
profile.round(1)

## 8. Purchase Pattern Analysis

In [ ]:
# Revenue by fuel type
fuel_rev = df.groupby('Fuel Type')['Revenue (INR)'].sum().sort_values(ascending=False)
fuel_rev.plot(kind='barh', figsize=(6,4), color='#4C72B0', title='Revenue by Fuel Type')
plt.gca().invert_yaxis(); plt.show()

In [ ]:
# Monthly revenue trend
monthly = df.set_index('Date').resample('ME')['Revenue (INR)'].sum()
monthly.plot(marker='o', figsize=(8,4), title='Monthly Revenue Trend')
plt.show()

In [ ]:
# Fuel type preference by segment
seg_fuel = (df.merge(features[['Dealer','Region','Segment_Label']], on=['Dealer','Region'])
              .groupby(['Segment_Label','Fuel Type'])['Revenue (INR)'].sum().unstack(fill_value=0))
seg_fuel_pct = seg_fuel.div(seg_fuel.sum(axis=1), axis=0) * 100
sns.heatmap(seg_fuel_pct.reindex(['High-Value','Mid-Value','Low-Value']), annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Fuel Type Preference by Segment (%)'); plt.show()

## 9. Save Results

In [ ]:
features.to_csv('customer_features_segmented.csv', index=False)
profile.to_csv('segment_profile.csv')
print('Saved.')